<div dir="rtl">

# الترم ٢ · الوحدة ٢: موديلات بسيطة

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman-abozarifa04/ai-for-kids-course/blob/main/notebooks/teens/term-2/05-simple-models.ipynb)

هنجرّب ٣ موديلات مشهورة بنفس خطوات scikit-learn: `fit` ← `predict` ← `score`.

| الموديل | بيعمل إيه | مثال |
|---|---|---|
| Linear Regression | يتوقع **رقم** | سعر شقة |
| Decision Tree | يصنّف بأسئلة نعم/لا | الطالب هينجح ولا لأ |
| K-Means | يجمّع حاجات شبه بعض | تقسيم العملاء |

</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score

rng = np.random.default_rng(7)

<div dir="rtl">

## ١. Linear Regression: نتوقع سعر شقة في القاهرة

البيانات خيالية: المساحة بالمتر، وعدد الأوض، والسعر بالألف جنيه.

</div>

In [ ]:
n = 120
area = rng.uniform(60, 250, n).round()
rooms = np.clip((area / 45 + rng.normal(0, 0.7, n)).round(), 1, 6)
price = (12 * area + 90 * rooms + rng.normal(0, 150, n)).round()   # بالألف جنيه
flats = pd.DataFrame({"area_m2": area, "rooms": rooms, "price_k_egp": price})
flats.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(flats[["area_m2", "rooms"]], flats["price_k_egp"],
                                                    test_size=0.25, random_state=0)
reg = LinearRegression().fit(X_train, y_train)
pred = reg.predict(X_test)
print(f"Average error: {mean_absolute_error(y_test, pred):,.0f} k EGP")
print("Coefficients:", dict(zip(X_train.columns, reg.coef_.round(1))), "Intercept:", round(reg.intercept_, 1))

<div dir="rtl">

**فهم المعادلة:** الموديل اتعلم إن كل متر زيادة بيزوّد السعر حوالي ١٢ ألف جنيه. ده **قريب جداً** من الرقم اللي استخدمناه في توليد البيانات. الموديل اكتشف القاعدة لوحده!

</div>

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(y_test, pred, color="#4F46E5")
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, "--", color="gray")
plt.xlabel("Real price (k EGP)")
plt.ylabel("Predicted price (k EGP)")
plt.title("Predicted vs real")
plt.show()

<div dir="rtl">

## ٢. Decision Tree: هل الطالب هينجح؟

دي نفس **شجرة القرار** اللي الأطفال رسموها بالورق في الحصة ٧، بس الكمبيوتر هو اللي بيختار الأسئلة.

</div>

In [ ]:
n = 400
study = rng.uniform(0, 6, n)
absences = rng.integers(0, 15, n)
sleep = rng.uniform(4, 9, n)
passed = ((study * 2 - absences * 0.5 + (sleep - 6) + rng.normal(0, 2.0, n)) > 2).astype(int)
students = pd.DataFrame({"study_hours": study.round(1), "absences": absences,
                         "sleep_hours": sleep.round(1), "passed": passed})

X = students[["study_hours", "absences", "sleep_hours"]]
y = students["passed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(f"Accuracy: {accuracy_score(y_test, tree.predict(X_test)):.0%}")

plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=X.columns, class_names=["fail", "pass"], filled=True, rounded=True, fontsize=9)
plt.show()

<div dir="rtl">

**اقرا الشجرة:** ابدأ من فوق. كل مربع فيه سؤال، يمين = لأ، شمال = أيوه. السؤال اللي فوق خالص هو **أهم** feature في نظر الموديل.

</div>

In [ ]:
pd.Series(tree.feature_importances_, index=X.columns).sort_values().plot.barh(color="#059669")
plt.title("Feature importance")
plt.show()

<div dir="rtl">

### ⚠️ Overfitting: الحفظ بدل الفهم
لو سبنا الشجرة تكبر من غير حدود، هتحفظ بيانات التدريب بالظبط وتغلط في الجديد.

هنستخدم **cross-validation**: نقسم البيانات ٥ أجزاء، وندرّب ونختبر ٥ مرات بأجزاء مختلفة، وناخد المتوسط. كده النتيجة بتبقى أثبت من اختبار واحد.

</div>

In [ ]:
from sklearn.model_selection import cross_validate

depths, train_acc, test_acc = [1, 2, 3, 5, 10, None], [], []
for depth in depths:
    cv = cross_validate(DecisionTreeClassifier(max_depth=depth, random_state=0), X, y, cv=5, return_train_score=True)
    train_acc.append(cv["train_score"].mean())
    test_acc.append(cv["test_score"].mean())
    print(f"max_depth={str(depth):>4}  train={train_acc[-1]:.0%}  test={test_acc[-1]:.0%}")

labels = [str(d) for d in depths]
plt.figure(figsize=(7, 4))
plt.plot(labels, train_acc, marker="o", label="train")
plt.plot(labels, test_acc, marker="o", label="test")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Overfitting: train keeps rising, test does not")
plt.show()

<div dir="rtl">

كل ما العمق يزيد، دقة التدريب بتطلع لحد **١٠٠٪**، لكن دقة الاختبار بتوصل لأعلى نقطة عند عمق متوسط وبعدين **بتنزل**. المسافة الكبيرة بين الخطين هي **overfitting**: زي الطالب اللي حافظ إجابات الامتحان اللي فات، ولما الأسئلة اتغيرت ما عرفش.

</div>

<div dir="rtl">

## ٣. K-Means: نقسّم عملاء محل

محل عنده بيانات عن العملاء: بيصرفوا كام في الشهر، وبييجوا كام مرة. عايز يقسّمهم مجموعات علشان يعمل عروض مختلفة.

</div>

In [ ]:
groups = [rng.normal([300, 2], [60, 0.6], (40, 2)),
          rng.normal([1500, 8], [200, 1.5], (40, 2)),
          rng.normal([800, 4], [120, 1.0], (40, 2))]
customers = pd.DataFrame(np.vstack(groups), columns=["monthly_spend_egp", "visits_per_month"]).clip(lower=0)

from sklearn.preprocessing import StandardScaler
scaled = StandardScaler().fit_transform(customers)    # نخلي الأعمدة على نفس المقياس
customers["segment"] = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(scaled)

plt.figure(figsize=(7, 4))
plt.scatter(customers["monthly_spend_egp"], customers["visits_per_month"], c=customers["segment"], cmap="Set2")
plt.xlabel("Monthly spend (EGP)")
plt.ylabel("Visits per month")
plt.title("Customer segments")
plt.show()
customers.groupby("segment").mean().round(1)

<div dir="rtl">

**ليه StandardScaler؟** المصاريف بالمئات والزيارات بالآحاد. من غير توحيد المقياس، K-Means هيبص على المصاريف بس.

</div>

<div dir="rtl">

## 🏋️ تمرين ١
استخدم موديل الشقق `reg` وتوقّع سعر شقة ١٥٠ متر و٣ أوض.

<details><summary>الحل</summary>

```python
flat = pd.DataFrame([[150, 3]], columns=["area_m2", "rooms"])
print(f"Predicted price: {reg.predict(flat)[0]:,.0f} k EGP")
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## 🏋️ تمرين ٢
جرّب K-Means بـ ٢ و ٤ مجموعات. أنهي عدد منطقي أكتر؟ (استخدم `inertia_`: كل ما تقل كل ما المجموعات أضيق.)

<details><summary>الحل</summary>

```python
for k in range(1, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(scaled)
    print(k, round(km.inertia_, 1))
# الانخفاض بيبقى كبير لحد k=3 وبعدين بيبطّأ ("الكوع")، يعني ٣ هو الأنسب.
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## ✅ لخّص
- **Regression** ← رقم. نقيس بـ متوسط الخطأ (MAE).
- **Classification** ← فئة. نقيس بـ الدقة (accuracy).
- **Clustering** ← مجموعات من غير إجابات.
- **Overfitting** = حفظ مش فهم. الحل: موديل أبسط، أو بيانات أكتر.

</div>